In [12]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import importlib
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any

import os
from collections import Counter
from queue import PriorityQueue

from pytket._tket.circuit import Circuit, OpType
from pytket.circuit.display import render_circuit_jupyter
from pytket.qasm import circuit_from_qasm
import json
import pytket
import numpy as np
from pytket.passes import EulerAngleReduction
from pytket import OpType   

from tket2.rewrite import default_ecc_rewriter, ECCRewriter, CircuitRewrite
from tket2.circuit import Tk2Circuit

In [2]:
invalid_file = "../test_files/invalid.qasm"

circ = circuit_from_qasm(invalid_file)
#with open(invalid_file, "r") as f:
#    circ = Circuit.from_dict(json.load(f))

eccs = default_ecc_rewriter()

In [9]:
class Work:
    num_cz: int
    num_gates: int
    circ: Circuit

    def __init__(self, circ: Circuit):
        self.num_cz = circ.n_gates_of_type(OpType.CZ)
        self.num_gates = circ.n_gates
        self.circ = circ

    @property
    def priority(self) -> (int, int):
        return (self.num_cz, self.num_gates)

    def __lt__(self, other):
        return self.priority < other.priority

def manual_badger(circ, eccs) -> bool:
    queue = PriorityQueue(maxsize=10)
    seen_hashes = set()
    initial_unitary = circ.get_unitary()

    queue.put(Work(circ))

    while not queue.empty():
        work = queue.get()
        circ_priority = work.priority
        circ_tk1 = work.circ
        circ = Tk2Circuit(circ_tk1)

        for rewrite in eccs.get_rewrites(circ):
            new_circ = circ.copy()

            if  rewrite.node_count_delta() > 0:
                continue

            rewrite.apply(new_circ)
            queue.put(Work(new_circ))

            circ_hash = circ.hash()
            if circ_hash in seen_hashes:
                continue
            seen_hashes.add(hash)

            if circ.node_count_delta() == 0:
                print("Found solution")
                return circ

            new_unitary = new_circ.get_unitary()
            if not np.allclose(new_unitary, initial_unitary):
                print("Unitary changed. See 'pre-circ.json' and 'post-circ.json'")
                json.dump(circ_tk1.to_dict(), open("pre-circ.json", "w"))
                json.dump(new_circ.to_tket1_json(), open("post-circ.json", "w"))
                json.dump(rewrite.replacement().to_tket1_json(), open("rewrite.json", "w"))
                return False

    print("The optimisation finished correctly")
    return True

def priority_diff(rewrite: CircuitRewrite) -> (int, int):
    return (rewrite.node_count_delta(), rewrite.node_count_delta())
